In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pycaret.regression import *
from xgboost import XGBRegressor
from pathlib import Path

In [81]:
PATH = Path('datasets')
df = pd.read_csv(PATH/"IndyCar_dataset_v24.csv")

In [82]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values("EventDate")

In [83]:
print(df[["DriverID", "NormalizedPositionFinish", "DRFAvg"]].groupby("DriverID").head(3).head(30))

    DriverID  NormalizedPositionFinish  DRFAvg
0       3608                      0.52     NaN
25      4407                      0.64     NaN
24      4401                      0.88     NaN
23      4276                      0.20     NaN
22      4236                      1.00     NaN
21      4216                      0.12     NaN
20      4215                      0.40     NaN
19      4144                      0.32     NaN
17      3813                      0.92     NaN
16      3811                      0.84     NaN
15      3736                      0.72     NaN
14      3682                      0.36     NaN
13      3680                      0.28     NaN
18      4021                      0.80     NaN
11      3672                      0.60     NaN
12      3675                      0.56     NaN
2       3620                      0.68     NaN
3       3622                      0.00     NaN
4       3625                      0.76     NaN
5       3628                      0.04     NaN
1       3616 

In [85]:
df.head()

,DriverName,DriverID,Rookie,DRFAvg,DTAvg,DTTAvg,DNFRate,TDNFRate,DriverElo,DriverTElo,...,BestLapSpeed,LapConsistency,PaceDeg,AvgPaceVSLeadPace,AvgPitTime,PitAvgTimeVSOverall,TrackAvgPitStops,FuelWindowEstimate,PositionFinish,NormalizedPositionFinish
0,Marco Andretti,3608,0,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14,0.52
25,Rubens Barrichello,4407,1,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17,0.64
24,Katherine Legge,4401,1,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23,0.88
23,Simon Pagenaud,4276,1,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,0.20
22,James Jakes,4236,0,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26,1.00


In [ ]:
#Pre-Qualy Non-ID Model
drop_cols = [
    "DriverName", "DriverID", "PositionStart", "TeamName", "TeamID", "CarEngine", "EngineID", "EventName", "Track", "EventTrackType",
    "EventDate", "EventDateFormatted", "EventID", "Era",
    "Status", "StatusID", "PositionFinish"
]

cutoff = df["EventDate"].quantile(0.95)
data = df[df["EventDate"] < cutoff].drop(columns=drop_cols)
data_unseen = df[df["EventDate"] >= cutoff].drop(columns=drop_cols)

print(data.corr(numeric_only=True)["NormalizedPositionFinish"].sort_values())

In [ ]:
#Pre-Qualy ID Model
drop_cols = [
    "DriverName", "PositionStart", "StartVsField","TeamName", "CarEngine", "EventName", "Track", "EventTrackType",
    "EventDate", "EventDateFormatted", "EventID", "Era",
    "Status", "StatusID", "PositionFinish",
    "TotalCautions", "TotalCautionLaps", "TotalLeadChanges",
    "AvgRaceSpeed", "FastestLapSpeed",
    "AvgStintLength", "StintVariance", "FirstStopLap", "FirstStopVSAvgFirstStop",
    "PittedCautions", "PitStops", "PittedCautionPCT", "TotalPitStops",
    "PitStrategy", #"PitStrategyID",
    "BestRacePosition", "WorstRacePosition", "LapsLed", "Top5Laps", "Top10Laps",
    "PositionVolatility", "position_lap1", "PositionsGainedLap1",
    "PositionsGainedPits", "PositionsGainedCaution",
    "BestLapSpeed", "LapConsistency", "PaceDeg", "AvgPaceVSLeadPace",
    "AvgPitTime", "PitAvgTimeVSOverall",
]

cutoff = df["EventDate"].quantile(0.95)
data = df[df["EventDate"] < cutoff].drop(columns=drop_cols)
data_unseen = df[df["EventDate"] >= cutoff].drop(columns=drop_cols)

print(data.corr(numeric_only=True)["NormalizedPositionFinish"].sort_values())

In [84]:
#Post-Qualy Model
drop_cols = [
    "DriverName", "TeamName", "CarEngine", "EventName", "Track", "EventTrackType",
    "EventDate", "EventDateFormatted", "EventID", "Era",
    "Status", "StatusID", "PositionFinish",
    "TotalCautions", "TotalCautionLaps", "TotalLeadChanges",
    "AvgRaceSpeed", "FastestLapSpeed",
    "AvgStintLength", "StintVariance", "FirstStopLap", "FirstStopVSAvgFirstStop",
    "PittedCautions", "PitStops", "PittedCautionPCT", "TotalPitStops",
    "PitStrategy", #"PitStrategyID",
    "BestRacePosition", "WorstRacePosition", "LapsLed", "Top5Laps", "Top10Laps",
    "PositionVolatility", "position_lap1", "PositionsGainedLap1",
    "PositionsGainedPits", "PositionsGainedCaution",
    "BestLapSpeed", "LapConsistency", "PaceDeg", "AvgPaceVSLeadPace",
    "AvgPitTime", "PitAvgTimeVSOverall",
]

cutoff = df["EventDate"].quantile(0.95)
data = data.fillna(data.median(numeric_only=True))
data_unseen = data_unseen.fillna(data_unseen.median(numeric_only=True))
#data = df[df["EventDate"] < cutoff].drop(columns=drop_cols)
#data_unseen = df[df["EventDate"] >= cutoff].drop(columns=drop_cols)

print(data.corr(numeric_only=True)["NormalizedPositionFinish"].sort_values())

DriverElo                      -3.935091e-01
DriverTTElo                    -3.688284e-01
TeamElo                        -3.252843e-01
DriverTElo                     -2.411139e-01
TeamTElo                       -2.389058e-01
TeamID                         -1.191421e-01
EngineTTElo                    -7.213333e-02
EngineElo                      -5.976131e-02
EngineTElo                     -4.367183e-02
TeamTrackAvgPitStops           -1.579384e-02
TeamTrackAvgPittedCautionPCT   -9.133241e-03
TrackID                        -2.446460e-03
TrackAvgPitStops               -2.420624e-03
EraID                          -2.990131e-15
FieldSize                      -1.204454e-15
TotalRaceLaps                  -1.971793e-16
TrackLength                     2.234818e-16
EventTrackTypeID                3.229748e-16
FuelWindowEstimate              4.303690e-04
TrackAvgSpeed                   4.470214e-04
TrackTypeAvgCautionLaps         4.732493e-04
TrackTypeAvgCautions            8.612582e-04
TrackAvgCa

In [86]:
df = df.drop(columns=drop_cols)

In [87]:
print(df.columns.tolist())

['DriverID', 'Rookie', 'DRFAvg', 'DTAvg', 'DTTAvg', 'DNFRate', 'TDNFRate', 'DriverElo', 'DriverTElo', 'DriverTTElo', 'DriverRitmo', 'PositionStart', 'StartVsField', 'TeamID', 'TRP', 'TTP', 'TeamDNFRate', 'TeamElo', 'TeamTElo', 'TeamRitmo', 'TeamTrackAvgPitStops', 'TeamTrackAvgPittedCautionPCT', 'EngineID', 'EngineElo', 'EngineTElo', 'EngineTTElo', 'TrackID', 'EventTrackTypeID', 'TrackAvgCautions', 'TrackAvgCautionLaps', 'TrackTypeAvgCautions', 'TrackTypeAvgCautionLaps', 'TrackAvgSpeed', 'EraID', 'FieldSize', 'TotalRaceLaps', 'TrackLength', 'TrackAvgPitStops', 'FuelWindowEstimate', 'NormalizedPositionFinish']


In [88]:
exp = setup(
    data=data, 
    target="NormalizedPositionFinish", 
    session_id=123, 
    fold_strategy="timeseries",
    data_split_shuffle=False,
    fold_shuffle=False
)

,Description,Value
0,Session id,123
1,Target,NormalizedPositionFinish
2,Target type,Regression
3,Original data shape,"(5608, 38)"
4,Transformed data shape,"(5608, 38)"
5,Transformed train set shape,"(3925, 38)"
6,Transformed test set shape,"(1683, 38)"
7,Numeric features,37
8,Preprocess,True
9,Imputation type,simple


In [ ]:
compare_models()

In [89]:
rf = create_model('rf')
rf_tune = tune_model(rf)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2533,0.0894,0.2989,0.0152,0.2072,1.2243
1,0.2531,0.0891,0.2985,0.0158,0.2064,1.1084
2,0.2430,0.0809,0.2845,0.1037,0.1963,1.0917
3,0.2406,0.0813,0.2852,0.1045,0.1967,1.0660
4,0.2461,0.0823,0.2868,0.0988,0.1976,1.0820
5,0.2311,0.0788,0.2807,0.1309,0.1946,1.0645
6,0.2341,0.0772,0.2778,0.1512,0.1893,0.9441
7,0.2375,0.0793,0.2816,0.1252,0.1918,1.0014
8,0.2321,0.0750,0.2738,0.1737,0.1874,1.0300


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2503,0.0852,0.2919,0.0612,0.2021,1.1911
1,0.2446,0.0833,0.2886,0.0803,0.1988,1.0490
2,0.2392,0.0801,0.2831,0.1125,0.1942,1.0504
3,0.2364,0.0791,0.2812,0.1292,0.1931,1.0174
4,0.2355,0.0754,0.2747,0.1735,0.1883,1.0116
5,0.2269,0.0749,0.2737,0.1735,0.1887,1.0205
6,0.2294,0.0743,0.2727,0.1824,0.1851,0.9197
7,0.2382,0.0789,0.2809,0.1292,0.1917,1.0113
8,0.2255,0.0726,0.2694,0.2002,0.1842,0.9854


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [97]:
predict_model(rf_tune);
predict_model(rf);
newpred2 = predict_model(rf_tune, data=data_unseen)
newpred2 = predict_model(rf, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,0.2173,0.0677,0.2603,0.2447,0.1772,0.9536


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,0.2228,0.0708,0.2661,0.2105,0.1828,1.0156


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,0.2269,0.0731,0.2704,0.1883,0.1839,0.9763


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,0.2328,0.0775,0.2783,0.1403,0.1899,1.0108


In [90]:
gbr = create_model('gbr')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2745,0.1080,0.3287,-0.1904,0.2250,1.2417
1,0.2629,0.0984,0.3137,-0.0869,0.2166,1.0772
2,0.2466,0.0867,0.2944,0.0398,0.2029,1.0890
3,0.2419,0.0850,0.2916,0.0635,0.2002,1.0297
4,0.2467,0.0844,0.2905,0.0758,0.1991,1.1047
5,0.2326,0.0814,0.2854,0.1017,0.1959,1.0065
6,0.2265,0.0749,0.2736,0.1766,0.1843,0.8497
7,0.2354,0.0796,0.2821,0.1217,0.1913,0.9642
8,0.2288,0.0751,0.2740,0.1728,0.1875,1.0037


In [91]:
gbr_tune = tune_model(gbr)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2519,0.0861,0.2935,0.0510,0.2035,1.2114
1,0.2507,0.0867,0.2944,0.0427,0.2034,1.1069
2,0.2440,0.0815,0.2855,0.0970,0.1966,1.0971
3,0.2401,0.0796,0.2821,0.1237,0.1946,1.0539
4,0.2422,0.0798,0.2826,0.1252,0.1945,1.0708
5,0.2314,0.0764,0.2764,0.1573,0.1913,1.0642
6,0.2327,0.0748,0.2736,0.1768,0.1867,0.9600
7,0.2420,0.0798,0.2825,0.1195,0.1933,1.0570
8,0.2303,0.0734,0.2709,0.1915,0.1860,1.0285


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [ ]:
predict_model(gbr);
#predict_model(gbr_tune);

In [ ]:
#newpred3 = predict_model(gbr_tune, data=data_unseen)
newpred3 = predict_model(gbr, data=data_unseen)

In [92]:
cat = create_model('catboost')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2621,0.0958,0.3094,-0.0552,0.2128,1.2153
1,0.2593,0.0945,0.3075,-0.0439,0.2120,1.0497
2,0.2482,0.0877,0.2961,0.0286,0.2034,1.0745
3,0.2418,0.0865,0.2941,0.0477,0.2028,1.0527
4,0.2466,0.0860,0.2932,0.0583,0.1997,1.0239
5,0.2294,0.0780,0.2792,0.1400,0.1926,1.0306
6,0.2301,0.0770,0.2775,0.1532,0.1873,0.8411
7,0.2375,0.0821,0.2865,0.0944,0.1941,0.9383
8,0.2351,0.0800,0.2828,0.1185,0.1925,1.0149


In [93]:
cat_tune = tune_model(cat)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2477,0.0850,0.2916,0.0632,0.2001,1.1510
1,0.2527,0.0892,0.2987,0.0149,0.2059,1.0891
2,0.2410,0.0820,0.2863,0.0920,0.1970,1.0565
3,0.2391,0.0807,0.2840,0.1117,0.1957,1.0493
4,0.2386,0.0782,0.2796,0.1438,0.1915,1.0243
5,0.2258,0.0750,0.2739,0.1721,0.1883,1.0058
6,0.2299,0.0754,0.2746,0.1709,0.1864,0.9341
7,0.2397,0.0806,0.2840,0.1103,0.1932,1.0028
8,0.2236,0.0721,0.2686,0.2052,0.1833,0.9525


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [ ]:
predict_model(cat_tune);
#predict_model(cat);

In [ ]:
newpred5 = predict_model(cat_tune, data=data_unseen)
#newpred5 = predict_model(cat, data=data_unseen)

In [94]:
lgbm = create_model('lightgbm')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2754,0.1065,0.3264,-0.1738,0.2245,1.3375
1,0.2624,0.0988,0.3143,-0.0906,0.2170,1.0713
2,0.2521,0.0930,0.3050,-0.0306,0.2094,1.0972
3,0.2474,0.0897,0.2995,0.0124,0.2059,1.0441
4,0.2477,0.0866,0.2943,0.0514,0.2006,1.0277
5,0.2326,0.0821,0.2865,0.0944,0.1968,0.9954
6,0.2326,0.0795,0.2819,0.1257,0.1901,0.8396
7,0.2364,0.0846,0.2908,0.0669,0.1974,0.9377
8,0.2303,0.0772,0.2778,0.1496,0.1892,0.9934


In [96]:
lgbm_tune = tune_model(lgbm)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2542,0.0856,0.2925,0.0571,0.2023,1.1958
1,0.2458,0.0830,0.2881,0.0832,0.1990,1.0598
2,0.2402,0.0808,0.2842,0.1051,0.1952,1.0690
3,0.2378,0.0805,0.2837,0.1135,0.1947,1.0053
4,0.2334,0.0750,0.2739,0.1784,0.1875,1.0047
5,0.2283,0.0754,0.2745,0.1685,0.1897,1.0363
6,0.2277,0.0741,0.2722,0.1848,0.1848,0.9193
7,0.2414,0.0807,0.2841,0.1093,0.1939,1.0322
8,0.2222,0.0715,0.2674,0.2120,0.1825,0.9555


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [ ]:
predict_model(lgbm_tune);
#predict_model(lgbm);

In [ ]:
newpred5 = predict_model(lgbm_tune, data=data_unseen)
#newpred5 = predict_model(lgbm, data=data_unseen)

In [ ]:
blend1 = blend_models([rf_tune, lgbm_tune])

In [ ]:
blend1_tune = tune_model(blend1)

In [ ]:
#predict_model(blend1_tune);
predict_model(blend1);

In [ ]:
#newpred5 = predict_model(blend1_tune, data=data_unseen)
newpred5 = predict_model(blend1, data=data_unseen)

In [ ]:
blend2 = blend_models([rf_tune, lgbm_tune, cat_tune])

In [ ]:
blend2_tune = tune_model(blend2)

In [ ]:
predict_model(blend2_tune);
#predict_model(blend2);

In [ ]:
newpred6 = predict_model(blend2_tune, data=data_unseen)
#newpred6 = predict_model(blend2, data=data_unseen)

In [ ]:
blend3 = blend_models([rf_tune, lgbm_tune, gbr])

In [ ]:
blend3_tune = tune_model(blend3)

In [ ]:
predict_model(blend3_tune);
#predict_model(blend3);

In [ ]:
newpred = predict_model(blend3_tune, data=data_unseen)
#newpred = predict_model(blend3, data=data_unseen)

In [ ]:
blend4 = blend_models([rf_tune, lgbm_tune, gbr, cat_tune])

In [ ]:
blend4_tune = tune_model(blend4)

In [ ]:
#predict_model(blend4_tune);
predict_model(blend4);

In [ ]:
#newpred = predict_model(blend4_tune, data=data_unseen)
newpred = predict_model(blend4, data=data_unseen)

In [ ]:
blend5 = blend_models([rf_tune, cat_tune])

In [ ]:
blend5_tune = tune_model(blend5)

In [ ]:
predict_model(blend5_tune);
#predict_model(blend5);

In [ ]:
newpred = predict_model(blend5_tune, data=data_unseen)
#newpred = predict_model(blend5, data=data_unseen)

In [ ]:
blend6 = blend_models([lgbm_tune, cat_tune])

In [ ]:
blend6_tune = tune_model(blend6)

In [ ]:
predict_model(blend6_tune);
#predict_model(blend6);

In [ ]:
newpred = predict_model(blend6_tune, data=data_unseen)
#newpred = predict_model(blend6, data=data_unseen)

In [ ]:
blend7 = blend_models([lgbm_tune, gbr_tune])

In [ ]:
blend7_tune = tune_model(blend7)

In [ ]:
predict_model(blend7_tune);
#predict_model(blend7);

In [ ]:
newpred = predict_model(blend7_tune, data=data_unseen)
#newpred = predict_model(blend7, data=data_unseen)

In [ ]:
blend8 = blend_models([gbr_tune, cat_tune])

In [ ]:
blend8_tune = tune_model(blend8)

In [ ]:
predict_model(blend8_tune);
#predict_model(blend8);

In [ ]:
newpred = predict_model(blend8_tune, data=data_unseen)
#newpred = predict_model(blend8, data=data_unseen)

In [80]:
save_model(blend6_tune, "indycar_rf_lgbm_gbr_prequaly_model_v2")

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(include=['DriverID', 'Rookie', 'DRFAvg',
                                              'DTAvg', 'DTTAvg', 'DNFRate',
                                              'TDNFRate', 'DriverElo',
                                              'DriverTElo', 'DriverTTElo',
                                              'DriverRitmo', 'TeamID', 'TRP',
                                              'TTP', 'TeamDNFRate', 'TeamElo',
                                              'TeamTElo', 'TeamRitmo',
                                              'TeamTrackAvgPitStops',
                                              'TeamTrackAvgPittedCautionPCT',
                                              'EngineID', 'En...
                  VotingRegressor(estimators=[('Light Gradient Boosting Machine',
                                               LGBMRegressor(bagging_fraction=0.7,
               

In [ ]:
save_model(lgbm_tune, "indycar_lgbm_postqualy_model_v4")